## Table of Contents

- **1. [Encoder-Decoder Network with a Fixed Context Vector](#1-encoder-decoder-network-with-a-fixed-context-vector)**
  - 1.1 [Tokenization and Embedding](#11-tokenization-and-embedding)
  - 1.2 [The Encoder](#12-the-encoder)
  - 1.3 [The Context Vector](#13-the-context-vector)
  - 1.4 [The Decoder](#14-the-decoder)
  - 1.5 [Objective of Training](#15-Objective-of-Training)
- **2. [Limitations of the Fixed Context Vector](#2-limitations-of-the-fixed-context-vector)**
  - 2.1 [Sequential Computation Prevents Parallelization](#21-sequential-computation-prevents-parallelization)
  - 2.2 [Long-Range Dependencies Are Difficult to Learn](#22-long-range-dependencies-are-difficult-to-learn)
- **3. [Additive Attention (Bahdanau et al., 2014)](#3-additive-attention-bahdanau-et-al-2014)**
  - 3.1 [The Context Vector](#31-the-context-vector)
  - 3.2 [The Alignment Model](#32-the-alignment-model)
  - 3.3 [Objective of Training](#33-Objective-of-Training)

## 1. Encoder-Decoder Network with a Fixed Context Vector

Neural machine translation is a machine translation  

Before 2017, if you wanted to translate a sentence from English to German, the standard approach was an encoder-decoder Recurrent Neural Network (RNN), including its more advanced variants, Long Short-Term Memory (LSTM) and Gated Recurrent Units (GRU) (Cho et al., 2014; Bahdanau et al., 2014). Thats because, in language grammatical structure or word order carriers meaning and the meaning of what comes after deponds on what comes before. "The dog bit the man" and "The man bit the dog" have identical words but opposite meaning.RNN's does this by accumulating previous hidden states a context. 

<div align="center">
<img src="images/seq2seq.png" width="800" style="display:block; margin:0 auto;">
<em><strong>Fig 1.</strong> The unrolled encoder-decoder architecture for neural machine translation from English to German. Source: Merity (2016)</em>
</div>

### 1.1 Tokenization and Embedding

To accurately learn the semantic and grammatical dependencies of a language, the source sentence $\mathbf{x}$ is initially tokenized into a sequence of tokens.

$$\mathbf{x} = (x_1, x_2, \ldots, x_{T_x}) \tag{1}$$

where $x_i \in \mathbb{R}^{K_x}$ and $K_x$ represents length of distinct source tokens that the model is being trained on. Each token is  represented as one-hot vector for the token is then mapped to a dense vector by an embedding layer before being read by the encoder as in Eq.2.

$$\phi(x_i) = W_e x_i, \quad \phi: \mathbb{R}^{K_x} \to \mathbb{R}^{n} \text{ linear and differentiable} \tag{2}$$

where $W_e \in \mathbb{R}^{n \times K_x}$ is the learned embedding matrix and $n$ is the embedding dimension.

In a real-world machine translation setting, the source vocabulary size $K_x$ is typically very large, while the embedding dimension $n$ is chosen to be much smaller ($n \ll K_x$). The embedding layer thus projects the sparse one-hot vector $x_i \in \mathbb{R}^{K_x \times 1}$ into a dense, low-dimensional vector $\phi(x_i) \in \mathbb{R}^{n \times 1}$ before it reaches the encoder. This makes $\phi(x_i)$ compact, shrinking each token's representation from $K_x \times 1$ to $n \times 1$. Whether it is also information-rich - capturing meaningful semantic and grammatical relationships - depends not on the embedding operation itself, but on $W_e$ being optimized during training.

### 1.2 The Encoder

The encoder is a nonlinear function $f_{enc}$, e.g. an RNN, that reads the embedded input sentence one token at a time. A encoder $f_{enc}$ is the same* function at every step $t$ and does not depend on $t$ or on the sentence length $T_x$, allowing one trained function to encode sentences of any length.

At each step $t$, it produces a hidden state $h_t$ - a fixed-size vector summarizing everything read so far, up to and including $x_t$ - as shown in the encoding sequence below (Bahdanau et al., 2014, §2.1).

$$h_0 \xrightarrow{} \phi(x_1) \xrightarrow{h_1} \phi(x_2) \xrightarrow{h_2} \phi(x_3) \xrightarrow{h_3} \cdots \xrightarrow{h_{T_x-1}} \phi(x_{T_x}) \xrightarrow{h_{T_x}}$$

where $h_0$ (typically 0 or a learned vector) initializes the recurrence before any token is read.

The encoder's hidden state $h_t$ is computed as a function of the current token's embedding $\phi(x_t)$ and the previous hidden state $h_{t-1}$. It is represented as:

$$h_t = f_{enc}(\phi(x_t), h_{t-1}), \quad f_{enc}: \mathbb{R}^{n} \times \mathbb{R}^n \to \mathbb{R}^n \text{ nonlinear and differentiable} \tag{3}$$

where $n$ is the number of hidden units. 
### 1.3 The Context Vector

After reading the entire sequence, the encoder's final hidden state $h_{T_x}$ is taken as a fixed-length numerical vector, called the context vector $c$ (Bahdanau et al., 2014, Eq. 1; Sutskever et al., 2014)

$$c = q(\{h_1, \ldots, h_{T_x}\}) = h_{T_x}, \quad q_{T_x}: (\mathbb{R}^n)^{T_x} \to \mathbb{R}^n \text{ nonlinear and differentiable} \tag{4}$$

Because $T_x$ varies with the length of the input sentence, $q$ is not a single fixed-arity function but a family $q = \{q_{T_x}\}_{T_x \geq 1}$, one map per possible sequence length, each reducing its (fixed-length) tuple of annotations to a single vector in $\mathbb{R}^n$. Equivalently, one can define $q$ on the union of finite-length tuples, $q: \bigcup_{k \geq 1} (\mathbb{R}^n)^k \to \mathbb{R}^n$, so that a single function correctly accepts input of any length. Either way, $n$ matches the encoder's hidden state size, and $q$ takes the *entire set* of annotations $\{h_1,\ldots,h_{T_x}\}$ - of variable size, depending on sentence length - and reduces it to one fixed-size vector.


### 1.4 The Decoder

The decoder is also a nonlinear, time-homogeneous function $f_{dec}$, e.g. an RNN, that predicts the next output token $y_t$ (the $t$-th word of the translation) based on the embedding of the latest generated token $\phi_y(y_{t-1})$, its previous hidden state $s_{t-1}$, and the context vector $c$, as shown in the decoding sequence below. As with $\phi$ in §1.1, $\phi_y: \mathbb{R}^{K_y} \to \mathbb{R}^n$ is a learned target-side embedding, so $f_{dec}$ never consumes a raw one-hot token.

Let $\mathbf{y} = (y_1, \ldots, y_{T_y})$ be a sequence of output tokens, where $y_i \in \mathbb{R}^{K_y}$ and $K_y$ is the target vocabulary size.

<div align="center">

$$
\begin{array}{ccccccccc}
 & & y_1 & & y_2 & & y_3 & & y_4 \\
 & & \big\uparrow & \searrow & \big\uparrow & \searrow & \big\uparrow & \searrow & \big\uparrow \\
s_0 & \xrightarrow{} & s_1 & \xrightarrow{} & s_2 & \xrightarrow{} & s_3 & \xrightarrow{} & s_4 \xrightarrow{} \cdots \\
 & & \big\uparrow & & \big\uparrow & & \big\uparrow & & \big\uparrow \\
 & & c & & c & & c & & c
\end{array}
$$

*__Fig. 2.__ Decoder recurrence with a single fixed context vector, initialized from $s_0$ (typically derived from the encoder's final state). Every decoder state $s_i$ is conditioned on the same $c$ (upward arrows), the one compressed summary of the entire source sentence produced once by the encoder. The horizontal arrows $s_i \to s_{i+1}$ carry the recurrent state forward, while the diagonal arrows show the previous output $y_{i-1}$ feeding back as input to $s_i$; each $s_i$ then predicts $y_i$. Because $c$ never changes across decoding steps, the decoder has no way to shift its focus to different parts of the source as it generates each output token.*
</div>

At each step $t$, the decoder's hidden state $s_t$ is updated according to:

$$s_t = f_{dec}(s_{t-1}, \phi_y(y_{t-1}), c), \quad f_{dec}: \mathbb{R}^n \times \mathbb{R}^{n} \times \mathbb{R}^n \to \mathbb{R}^n \text{ nonlinear and differentiable} \tag{5}$$

matching the same hidden size $n$ used by the encoder, so that $c$ and $s_t$ are compatible when combined.

The decoder does not predict $y_t$ directly from $s_t$ alone. It defines a probability distribution over the translation $\mathbf{y}$ by decomposing the joint probability into ordered conditionals using the chain rule of probability (Bahdanau et al., 2014, Eq. 2):

$$p(\mathbf{y}) = \prod_{t=1}^{T_y} p(y_t \mid \{y_1, \ldots, y_{t-1}\}, c) \tag{6}$$

$s_t$ is only a memory vector, not a probability distribution, so it cannot by itself tell us which word comes next. An additional step is needed to turn that memory into an actual estimate over the vocabulary: a function $g$ takes $s_t$, together with $\phi_y(y_{t-1})$ and $c$, and outputs a probability for every possible word, from which $y_t$ is chosen (Bahdanau et al., 2014, Eq. 3).

$$p(y_t \mid \{y_1, \ldots, y_{t-1}\}, c) = g(\phi_y(y_{t-1}), s_t, c), \quad g: \mathbb{R}^{n} \times \mathbb{R}^n \times \mathbb{R}^n \to \mathbb{R}^{K_y} \text{ nonlinear and differentiable} \tag{7}$$

### 1.5 Objective of Training:

Given a dataset of $N$ source-target sentence pairs $\{(\mathbf{x}^{(1)}, \mathbf{y}^{(1)}), \ldots, (\mathbf{x}^{(N)}, \mathbf{y}^{(N)})\}$, training seeks:

$$\theta^* = \arg\max_{\theta} \sum_{i=1}^{N} \log p_\theta(\mathbf{y}^{(i)} \mid \mathbf{x}^{(i)}) \tag{8}$$

where $p_\theta(\mathbf{y}^{(i)} \mid \mathbf{x}^{(i)})$ is the same conditional distribution written as $p(\mathbf{y})$ in Eq. 6, with dependence on $\mathbf{x}^{(i)}$ carried implicitly through the context vector $c$.

We use the log-probability instead of the raw probability because $p(\mathbf{y} \mid \mathbf{x})$ is a product of many conditional terms (Eq. 6), each less than 1 — multiplying them risks numerical underflow, while their log becomes a sum:

$$\log p(\mathbf{y} \mid \mathbf{x}) = \sum_{t=1}^{T_y} \log p(y_t \mid \{y_1, \ldots, y_{t-1}\}, c) \tag{9}$$

Since optimizers minimize rather than maximize, we negate this to get the cross-entropy loss:

$$\mathcal{L} = -\sum_{t=1}^{T_y} \log p(y_t \mid \{y_1, \ldots, y_{t-1}\}, c) \tag{10}$$

Minimizing $\mathcal{L}$ is thus equivalent to maximizing $p(\mathbf{y} \mid \mathbf{x})$.

## 2. Limitations of the Fixed Context Vector

Although this training procedure lets the encoder and decoder jointly learn to produce translations, the underlying RNN architecture that computes $f_{enc}$, $c$, and $f_{dec}$ imposes structural constraints that no amount of training can remove. In particular, the same recurrence that allows $h_t$ and $s_t$ to be computed step by step is also what limits the architecture. This creates two problems:

### 2.1 Sequential Computation Prevents Parallelization

Since both the encoder's hidden state $h_t = f_{enc}(\phi(x_t), h_{t-1})$ (Eq. 3) and the decoder's hidden state $s_t = f_{dec}(s_{t-1}, y_{t-1}, c)$ (Eq. 5) depend on their respective previous hidden state, neither the encoder nor the decoder can compute positions within a sequence simultaneously (Vaswani et al., 2017). The decoder's hidden state still cannot be computed until the prior state exists, so this recurrence blocks parallelization on both sides of the model. This becomes especially limiting when training on large amounts of data under constrained computation or memory resources.

### 2.2 Long-Range Dependencies Are Difficult to Learn

Standard RNNs, and even their gated variants - LSTM (Hochreiter & Schmidhuber, 1997) and GRU (Cho et al., 2014) - compress all preceding context into a single fixed-size hidden state. Information relevant at position $t$ must survive being repeatedly overwritten across every intermediate step to influence a distant position $t+k$, and ultimately to survive all the way into the context vector $c$ (Eq. 4). If information about an early token is not retained in $c$, the decoder has no way to recover it, resulting in a loss of information in the output sequence. Hochreiter et al. (2001) showed that this leads to vanishing/exploding gradients, making it difficult for RNNs to learn dependencies between positions that are far apart in the sequence. If some of the information in the compressed encoded vector is missing, how much can we rely on each token at the source sequence to predict a particular output token remains in question.

## 3. Additive Attention (Bahdanau et al., 2014)

Bahdanau et al. (2014) addressed the second problem by introducing an attention mechanism: at each decoding step $i$, rather than relying on a single fixed vector $c$ that compresses the whole source sentence, the decoder computes a fresh context vector $c_i$ as a weighted sum over all encoder hidden states — which Bahdanau et al. termed annotations, since each one is retained and made individually available for the decoder to consult (Bahdanau et al., 2014, Eq. 5).

### 3.1 The Context Vector

This allows the decoder to draw on different parts of the source sentence at each step, rather than relying on one fixed summary:

$$c_i = \sum_{j=1}^{T_x} \alpha_{ij} h_j \tag{11}$$

The weight $\alpha_{ij}$ assigned to each annotation $h_j$ is obtained by normalizing a set of scores with a softmax, so that the weights are non-negative and sum to 1 (Bahdanau et al., 2014, Eq. 6):

$$\alpha_{ij} = \frac{\exp(e_{ij})}{\sum_{k=1}^{T_x} \exp(e_{ik})} \tag{12}$$

Since the weights $\alpha_{ij}$ are non-negative and sum to 1 across $j$, the sum $c_i = \sum_j\alpha_{ij}h_j$ (Eq. 11) can be understood as an expected annotation, where the expectation is taken over possible alignments - $\alpha_{ij}$ playing the role of the probability that target word $y_i$ is aligned to (translated from) source word $x_j$ (Bahdanau et al., 2014, §3.1). Rather than committing to one single source word as the correct alignment, the model blends information from all source positions, weighted by how relevant each one currently is.

<div align="center">

$$
\begin{array}{ccccccccc}
y_1 & & y_2 & & y_3 & & y_4 & & \\
\big\uparrow & \searrow & \big\uparrow & \searrow & \big\uparrow & \searrow & \big\uparrow & \searrow & \\
s_1 & \xrightarrow{} & s_2 & \xrightarrow{} & s_3 & \xrightarrow{} & s_4 & \xrightarrow{} & \cdots \\
\big\uparrow & & \big\uparrow & & \big\uparrow & & \big\uparrow & & \\
c_1 & & c_2 & & c_3 & & c_4 & &
\end{array}
$$

*__Fig. 3.__ Decoder recurrence with per-step soft attention. Unlike the fixed-$c$ architecture, each decoder state $s_i$ receives its own context vector $c_i$ (upward arrows), computed as a weighted sum over all encoder annotations $h_j$. The horizontal arrows $s_i \to s_{i+1}$ carry the recurrent state forward, while the diagonal arrows show the previous output $y_{i-1}$ feeding back as input to $s_i$; each $s_i$ then predicts $y_i$. The weights $\alpha_{ij}$ that determine $c_i$ are recomputed at every step, so the source information the decoder draws on shifts as decoding proceeds.*

</div>

### 3.2 The Alignment Model

The attention score $e_{ij}$ indicates how relevant source token $x_j$ is for predicting target token $y_i$. Each raw score $e_{ij}$ is computed by an alignment model $a$, comparing the decoder's previous hidden state $s_{i-1}$ against a single encoder annotation $h_j$ (Bahdanau et al., 2014, §3.1):

$$e_{ij} = a(s_{i-1}, h_j) = v_a^\top \tanh(W_a s_{i-1} + U_a h_j) \tag{13}$$

where $W_a$, $U_a$, and $v_a$ are learned weight matrices and vector. $a$ is parametrized as a small feedforward neural network, jointly trained end-to-end with the rest of the model — it is not hand-designed, and its behavior as a compatibility score between source and target positions emerges purely from training on translation data (Bahdanau et al., 2014, §3.1).

### 3.3 Objective of Training

Bahdanau attention leaves the training objective unchanged in form: since $c_i$ (Eq. 11) simply replaces $c$ (Eq. 4) inside $g$ (Eq. 7), the model is still trained by minimizing the same negative log-likelihood loss (compare Eq. 10), now conditioned on a per-step context vector rather than a single fixed one:

$$\mathcal{L} = -\sum_{i=1}^{T_y} \log p(y_i \mid \{y_1, \ldots, y_{i-1}\}, c_i) \tag{14}$$

In [2]:
import sys
!{sys.executable} -m pip install datasets scikit-learn sentencepiece sacrebleu


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from datasets import load_dataset
from sklearn.model_selection import train_test_split
import pandas as pd

c:\Users\Sai\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
dataset=load_dataset("ai4bharat/samanantar","te",split="train")
small_dataset=dataset.select(range(50_000))

In [6]:
split=small_dataset.train_test_split(test_size=0.05,seed=42)
train_data,val_data=split["train"],split["test"]

In [9]:
print(len(train_data),len(val_data))
print(train_data[:10])

47500 2500
{'idx': [40315, 33644, 9452, 10051, 272, 31498, 32334, 49053, 7342, 44665], 'src': ['They are very dangerous.', 'The CPM and other Left parties, as also the Trinamool Congress, the BJP and the Congress opposed the bandh.', 'Traffic stranded', 'Earthquake in Japan', 'Ganga Dialouges, which is a series of discussions with eminent personalities, started with renowned author and researcher Rajiv Malhotra in conversation with Shri Satyanarayana Dasa, a Vaishnava scholar', 'Goa is a fine tourist spot.', 'Good results.', 'After lunch visiting of Srinivas Mangarammn Kanipak kshetralu', 'I replied: Your Honor, I should have been classified as a minister.', 'Yes, its been that long.'], 'tgt': ['వీరు చాలా ప్రమాదకారులు.', 'ఈ బంద్\u200cకు సిపిఎం, ఇతర వామపక్ష పార్టీలు, వైసిపి, కాంగ్రెస్\u200cలు మద్దతు ప్రకటించాయి.', 'స్తంభించిపోయిన ట్రాఫిక్\u200c', 'జపాన్\u200cలో భూకంపం భారీ విధ్వంసం', 'వైష్ణవ పండితుడు శ్రీ సత్యనారాయణ దాసతో ప్రముఖ రచయిత, పరిశోధకుడు రాజీవ్ మల్హోత్రా నిర్వహించిన గోష్ఠితో ఈ 

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F


# Tokenize and build vocabularies

Whitespaces tokenization badly undercounts Telugu(agglutinative, so a word-level vocabulary explodes and rare inflection becomes <unk>) and it treats English morphology(walk/walked)


**The Transformer's proposal**

Vaswani et al. (2017) proposed removing recurrence entirely and relying solely on attention: "In this work we propose the Transformer, a model architecture eschewing recurrence and instead relying entirely on an attention mechanism to draw global dependencies between input and output". This directly resolves both problems:

- *Parallelization*: since no position's representation depends on first computing another position's, all positions can be processed simultaneously.
- *Path length*: self-attention connects any two positions with O(1) sequential operations, compared to O(n) for a recurrent layer (Vaswani et al., 2017, Table 1), making long-range dependencies no harder to learn than short-range ones.


**References**

- Bahdanau, D., Cho, K., & Bengio, Y. (2014). *Neural Machine Translation by Jointly Learning to Align and Translate*. arXiv:1409.0473.
- Cho, K., et al. (2014). *Learning Phrase Representations using RNN Encoder-Decoder for Statistical Machine Translation*. arXiv:1406.1078.
- Hochreiter, S., & Schmidhuber, J. (1997). Long short-term memory. *Neural Computation*, 9(8), 1735–1780.
- Hochreiter, S., Bengio, Y., Frasconi, P., & Schmidhuber, J. (2001). *Gradient Flow in Recurrent Nets: The Difficulty of Learning Long-Term Dependencies*.
- Merity, S. (2016). *Peeking into the Neural Network Architecture used for Google's Neural Machine Translation*. Retrieved from https://smerity.com/articles/2016/google_nmt_arch.html
- Sutskever, I., Vinyals, O., & Le, Q. V. (2014). *Sequence to Sequence Learning with Neural Networks*. In *Advances in NeurIPS 27*.
- Vaswani, A., et al. (2017). *Attention Is All You Need*. In *Advances in NeurIPS 30*.